# Dreamer V1 — gated seed-2 100k-environment-step pilot

This notebook targets one independent seed-2 baseline on `cartpole_swingup`:
50,000 decision steps and 100,000 environment steps. The 300k/330k seed-0
overnight run remains separate. Seed 1 is treated as a provisional match
because its raw config/checkpoint files were not supplied; it is not used to
claim a statistically verified three-seed result.


In [ ]:
import base64
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

SESSION_STARTED = time.time()
SEED2_RUN_ID = 'seed2_matching_20261025_01'
SEED2_ROOT = Path('/kaggle/working') / f'dreamer_{SEED2_RUN_ID}'
SEED2_ROOT.mkdir(parents=True, exist_ok=True)
RUNNER = SEED2_ROOT / 'kaggle_pytorch_pilot.py'
RUNNER.write_bytes(base64.b64decode('...'))
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['MUJOCO_GL'] = 'egl'
os.environ['DREAMER_RUN_ID'] = SEED2_RUN_ID
os.environ['DREAMER_PILOT_ROOT'] = str(SEED2_ROOT)
print('run_id:', SEED2_RUN_ID)
print('root:', SEED2_ROOT)
print('python:', sys.executable)

ROOT = SEED2_ROOT


In [ ]:
import importlib.util
import subprocess
import sys

install_log = SEED2_ROOT / 'dm_control_no_deps_install.log'
if importlib.util.find_spec('dm_control') is None:
    command = [
        sys.executable, '-m', 'pip', 'install', '--no-deps',
        'dm-env==1.6', 'mujoco==3.3.7', 'dm-control==1.0.34',
        'glfw==2.11.0', 'pyprind==2.11.3',
    ]
    with install_log.open('w', encoding='utf-8') as stream:
        result = subprocess.run(command, stdout=stream,
                                stderr=subprocess.STDOUT, text=True)
    print('dm-control install returncode:', result.returncode)
    print('install log:', install_log)
    print(install_log.read_text(encoding='utf-8'))
    if result.returncode:
        raise RuntimeError('fixed dm-control --no-deps install failed')
else:
    install_log.write_text('dm_control already visible; install skipped.\n', encoding='utf-8')
    print('dm_control already visible; fixed install skipped.')
print('dm_control spec:', importlib.util.find_spec('dm_control'))


## Seed-1 provenance gate

The teammate reports that seed 1 was run from the repository's seed-0 100k
notebook with only the training seed changed to 1. The attached aggregate
summary confirms the task, ten evaluation seeds, and seed-1 summary values,
but does not independently prove every training configuration field. This
cell records that limitation and lets the independent seed-2 run proceed
with the repository notebook as the configuration source of truth.


In [ ]:
import csv
import hashlib
import json
from pathlib import Path

seed0 = {
    "task": "cartpole_swingup", "n_steps": 50000,
    "action_repeat": 2, "batch_size": 16, "batch_length": 50,
    "prefill_decision_steps": 1000, "train_every": 1000,
    "train_steps_per_itr": 100, "replay_capacity_decision_transitions": 60000,
}
summary = {
    "task": "cartpole_swingup",
    "seeds": list(range(1000, 1010)),
    "dreamer_mean_return": 227.0930596746478,
    "dreamer_sample_std_return": 29.473746946933314,
    "initial_mean_return": 71.937023100428,
    "initial_sample_std_return": 4.822296391089559,
    "random_mean_return": 45.15436797873804,
    "random_sample_std_return": 27.32074936914384,
    "paper_reproduction_claim": False,
    "source_summary_sha256": "8a9d51bcc040bf3c2368eda9f8f97e94cfc8f11f291496273c9b02821a82a7ef",
}
cfg = {
    "seed": 1,
    "task": "cartpole_swingup",
    "n_steps": 50000,
    "environment_steps": 100000,
    "action_repeat": 2,
    "batch_size": 16,
    "batch_length": 50,
    "prefill_decision_steps": 1000,
    "train_every": 1000,
    "train_steps_per_itr": 100,
    "replay_capacity_decision_transitions": 60000,
    "provenance": "teammate_reported_same_repository_notebook_only_seed_changed",
}
errors = []
if int(cfg.get('seed', -1)) != 1: errors.append(f"seed={cfg.get('seed')}, expected 1")
if cfg.get('task') != seed0['task']: errors.append('task mismatch')
for key in ('action_repeat', 'batch_size', 'batch_length', 'prefill_decision_steps', 'train_every', 'train_steps_per_itr', 'replay_capacity_decision_transitions'):
    if key in cfg and int(cfg[key]) != int(seed0[key]): errors.append(f'{key} mismatch')
actual_env = summary.get('actual_environment_steps') or summary.get('environment_steps') or cfg.get('environment_steps')
if actual_env is not None and int(actual_env) != 100000: errors.append(f'actual environment steps={actual_env}, expected 100000')
if summary.get('task') != seed0['task']: errors.append('summary task mismatch')
if summary.get('seeds') != list(range(1000, 1010)): errors.append('summary evaluation seeds mismatch')
gate = {
    'status': 'provisional_match' if not errors else 'blocked_mismatch',
    'errors': errors,
    'training_seed': cfg.get('seed'),
    'actual_environment_steps': actual_env,
    'evaluation_rows': 10,
    'raw_config_present': False,
    'raw_eval_csv_present': False,
    'checkpoint_present': False,
    'evidence_level': 'user_attested_repository_notebook_plus_aggregate_summary',
    'repository_notebook_source': 'notebooks/kaggle_dreamer_v1_pytorch_cartpole_pilot_e8b9486.ipynb',
    'upstream_commit': 'ccea6ae4a397a94c328891bd78e81d52dd156cb6',
    'rlpyt_commit': 'f04f23db1eb7b5915d88401fca67869968a07a37',
    'summary_sha256': summary['source_summary_sha256'],
}
(ROOT / 'seed1_match_gate.json').write_text(json.dumps(gate, indent=2) + '\n')
print(json.dumps(gate, indent=2, sort_keys=True))
if errors: raise RuntimeError('Seed-1 evidence gate failed: ' + '; '.join(errors))


In [ ]:
def run_child(phase, *args, child_root=SEED2_ROOT, timeout=None):
    child_root.mkdir(parents=True, exist_ok=True)
    log_path = child_root / f'{phase}_subprocess.log'
    env = os.environ.copy()
    env.update({
        'DREAMER_RUN_ID': SEED2_RUN_ID,
        'DREAMER_PILOT_ROOT': str(child_root),
        'CUDA_VISIBLE_DEVICES': '0',
        'MUJOCO_GL': 'egl',
    })
    command = [sys.executable, str(RUNNER), '--phase', phase, *map(str, args)]
    started = time.time()
    with log_path.open('w', encoding='utf-8') as stream:
        proc = subprocess.Popen(command, env=env, stdout=stream,
                                stderr=subprocess.STDOUT, text=True)
        while proc.poll() is None:
            elapsed = time.time() - started
            print(f'[{phase}] elapsed={elapsed/60:.1f} min log_bytes={log_path.stat().st_size if log_path.exists() else 0}', flush=True)
            if timeout is not None and elapsed > timeout:
                proc.terminate()
                raise TimeoutError(f'{phase} exceeded {timeout}s')
            time.sleep(30)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, command)
    print(f'[{phase}] complete in {(time.time()-started)/60:.1f} min')


## Persistence, CPU audit, and GPU/debug gates

The persistence probe is repeated in this independent run before training.
The fixed `dm-control --no-deps` route is reused only if the package is not
visible. The runner then checks CUDA/EGL and ten real Dreamer updates.


In [ ]:
run_child('preflight', '--seed', '2', timeout=15*60)
run_child('debug', '--seed', '2', timeout=15*60)


## Independent seed-2 100k training

This is a fresh seed-2 run: 50,000 decisions and 100,000 environment steps.
The formal seed-0 100k settings remain fixed: batch size 16, sequence length
50, prefill 1,000 decisions, train_every 1,000, train_steps 100, and replay
capacity 60,000 decision transitions. No other seed's weights are loaded.


In [ ]:
run_child('train', '--seed', '2', '--n-steps', '50000', timeout=90*60)
final_checkpoint = ROOT / 'train_run' / 'run_0' / 'params.pkl'
initial_checkpoint = ROOT / 'initial_policy.pkl'
run_child('eval', '--seed', '2', '--checkpoint', final_checkpoint,
          '--initial-checkpoint', initial_checkpoint, timeout=25*60)
run_child('export_minimal', timeout=10*60)
print('final checkpoint:', final_checkpoint)
print('initial checkpoint:', initial_checkpoint)


## Output and shutdown contract

Before stopping the session, download and hash the independent seed-2 run:
config, train summary, metrics, progress, initial/final checkpoints,
eval_episodes.csv, summary, logs, and the minimal bundle. Do not compute a
three-seed summary until the seed-1 gate and seed-2 CSV are independently
verified.
